In [2]:
import pandas as pd

df = pd.read_csv('/content/AB_NYC_2019.csv')

df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [ ]:
df.shape

(48895, 16)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48884 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              48884 non-null  int64         
 1   name                            48884 non-null  object        
 2   host_id                         48884 non-null  int64         
 3   host_name                       48884 non-null  object        
 4   neighbourhood_group             48884 non-null  object        
 5   neighbourhood                   48884 non-null  object        
 6   latitude                        48884 non-null  float64       
 7   longitude                       48884 non-null  float64       
 8   room_type                       48884 non-null  object        
 9   price                           48884 non-null  int64         
 10  minimum_nights                  48884 non-null  int64         
 11  number_

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isnull().sum()

,0
id,0
name,16
host_id,0
host_name,21
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


In [12]:
df['name'] = df['name'].fillna('Unknown')
df['host_name'] = df['host_name'].fillna('Unknown')

In [13]:
df['last_review'] = df['last_review'].fillna('No Reviews')
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

In [14]:
df.isnull().sum()

,0
id,0
name,0
host_id,0
host_name,0
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


In [ ]:
df.dtypes

,0
id,int64
name,object
host_id,int64
host_name,object
neighbourhood_group,object
neighbourhood,object
latitude,float64
longitude,float64
room_type,object
price,int64


In [21]:
df['last_review'] = pd.to_datetime(
    df['last_review'],
    errors='coerce'
)

In [ ]:
df['last_review'].dtype

dtype('<M8[ns]')

In [ ]:
print("Price <= 0:", (df['price'] <= 0).sum())
print("Minimum nights <= 0:", (df['minimum_nights'] <= 0).sum())
print("Availability outside 0-365:", ((df['availability_365'] < 0) | (df['availability_365'] > 365)).sum())
print("Latitude outside valid range:", ((df['latitude'] < -90) | (df['latitude'] > 90)).sum())
print("Longitude outside valid range:", ((df['longitude'] < -180) | (df['longitude'] > 180)).sum())

Price <= 0: 11
Minimum nights <= 0: 0
Availability outside 0-365: 0
Latitude outside valid range: 0
Longitude outside valid range: 0


In [16]:
# Remove listings with invalid prices
df = df[df['price'] > 0].copy()

In [17]:
print("Rows and columns:", df.shape)
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df['id'].duplicated().sum())

Rows and columns: (48884, 16)
Total missing values: 0
Duplicate rows: 0
Duplicate IDs: 0


In [18]:
# Before vs After Cleaning Summary

before_after = pd.DataFrame({
    'Metric': [
        'Rows',
        'Columns',
        'Missing values',
        'Duplicate rows',
        'Duplicate IDs',
        'Dtype accuracy'
    ],
    'Before Cleaning': [
        48895,
        16,
        20141,
        0,
        0,
        'Needs correction'
    ],
    'After Cleaning': [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df['id'].duplicated().sum(),
        'Correct'
    ]
})

before_after

,Metric,Before Cleaning,After Cleaning
0,Rows,48895,48884
1,Columns,16,16
2,Missing values,20141,0
3,Duplicate rows,0,0
4,Duplicate IDs,0,0
5,Dtype accuracy,Needs correction,Correct


In [19]:
numeric_columns = ['price', 'minimum_nights', 'number_of_reviews',
                   'reviews_per_month', 'calculated_host_listings_count',
                   'availability_365']

outlier_summary = {}

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()

    outlier_summary[column] = outliers

outlier_summary

{'price': np.int64(2972),
 'minimum_nights': np.int64(6605),
 'number_of_reviews': np.int64(6018),
 'reviews_per_month': np.int64(3309),
 'calculated_host_listings_count': np.int64(7073),
 'availability_365': np.int64(0)}

### Outlier Handling

The IQR method was used to identify potential outliers in the numerical variables. The flagged observations were retained because unusually high values may represent legitimate Airbnb listings rather than data-entry errors. No observations were removed solely because they were identified as IQR outliers.

In [ ]:
df = df[df['price'] > 0]

In [ ]:
df.shape

(48884, 16)

In [ ]:
df['id'].duplicated().sum()

np.int64(0)

In [15]:
print("Rows and columns:", df.shape)
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df['id'].duplicated().sum())

Rows and columns: (48895, 16)
Total missing values: 0
Duplicate rows: 0
Duplicate IDs: 0


In [10]:
df['last_review'] = df['last_review'].fillna(pd.Timestamp('1900-01-01'))

In [11]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 10089


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48884 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              48884 non-null  int64         
 1   name                            48884 non-null  object        
 2   host_id                         48884 non-null  int64         
 3   host_name                       48884 non-null  object        
 4   neighbourhood_group             48884 non-null  object        
 5   neighbourhood                   48884 non-null  object        
 6   latitude                        48884 non-null  float64       
 7   longitude                       48884 non-null  float64       
 8   room_type                       48884 non-null  object        
 9   price                           48884 non-null  int64         
 10  minimum_nights                  48884 non-null  int64         
 11  number_

## Data Cleaning Summary

The dataset was inspected and cleaned to improve data quality and consistency.

- Missing values in `name` and `host_name` were replaced with "Unknown".
- Missing values in `reviews_per_month` were replaced with 0 because these listings had no recorded reviews.
- Missing review dates were handled using a placeholder date.
- `last_review` was converted from object format to datetime format.
- Listings with invalid prices (price <= 0) were removed.
- No duplicate rows were found.
- No duplicate listing IDs were found.
- Latitude, longitude, minimum nights and availability values were checked and found to be within valid ranges.

After cleaning, the dataset contains 48,884 rows and 16 columns with no remaining missing values.

In [ ]:
df.to_csv('AB_NYC_2019_cleaned.csv', index=False)

In [ ]:
import os

os.path.exists('AB_NYC_2019_cleaned.csv')

True

In [ ]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,1900-01-01,0.00,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [ ]:
df.isnull().sum().sum(), df.duplicated().sum(), df['id'].duplicated().sum()

(np.int64(0), np.int64(0), np.int64(0))